In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.decomposition import PCA
import sys
import argparse
import os
import glob
import tqdm
from sklearn.decomposition import PCA
import numpy as np
 
import utils_report as ru
from utils_rnn import (
    plot_cumulative_variance,
    plot_pca_3d,
    plot_pca_2d,
    plot_all_agents_projected_2d,
    plot_all_agents_projected_3d,
    plot_rnn_vs_arena_distances,
    plot_behavior_and_rnn_state_trajectories,
    perform_pca_analysis_rnn_obs_mask,
)
from utils_behavior import (
    bin_agent_size,
    plot_behavior_densities_1d,
)
from utils_features import (
    get_rnn_state_deltas,
    calculate_column_feature_correlations,
    compute_clumpiness_scores_df,
    dynamic_knn_clumpiness,
    knn_distance_analysis,
    calculate_entropy,
    add_attention_entropy_columns,
)

import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import rcParams

rcParams["pdf.fonttype"] = 42  # Use Type 42 (TrueType) fonts to save text as text
rcParams["ps.fonttype"] = 42  # For PostScript as well, if needed
import seaborn as sns

# print("numpy version:", np.__version__)
# print("pandas version:", pd.__version__)

analyses_to_run = [
	"behavior_densities_1d",
	"pca_rnn_obs_mask",
	"pca_individual_agent",
	"pca_pooled_agent",
	"rnn_clumpiness_knn",
	"rnn_delta_distribution",
	"rnn_behavior_viz",
	"features_viz",
	"decoding_legacy",
	"decoding_modular",
	"rnn_delta_corr",
	"attn_entropy_corr",
	"umap_bool_features",
	"umap_non_bool_features",
	]

## Load data

In [ ]:
BASE_PATH = "./"
# SUB_PATH = "results/rmappo-MultiAgentFishEnv-3MSensorDropout5xGRU/20250721_162614/outputs/MAFish_neural_20250721_162614_07PhNBro_agg_flattened.pkl"
# SUB_PATH = "results/rmappo-MultiAgentForagingEnv-check/20250808_153214_1_bao_efp_0.05_vd_0.002_fd_10/outputs/MAZFish_neural_20250808_153214_bao_efp_0.05_vd_0.002_fd_10_agg_flattened.pkl"
SUB_PATH = "results/rmappo-MultiAgentForagingEnv-check/20250909_140455_1_bao_vd_0.003_fd_10_action_noise_0.1/outputs/MAZFish_neural_20250909_140455_bao_vd_0.003_fd_10_action_noise_0.1_agg_flattened.pkl"

flat_pkl_file = f"{BASE_PATH}/{SUB_PATH}"
OUTPUTS_FOLDER = os.path.dirname(flat_pkl_file)
# If not writeable, then save in current directory
if not os.access(OUTPUTS_FOLDER, os.W_OK):
    print(
        f"Outputs folder {OUTPUTS_FOLDER} is not writable, using current directory instead."
    )
    OUTPUTS_FOLDER = "./"
print(f"Using outputs folder: {OUTPUTS_FOLDER}")


dff = pd.read_pickle(flat_pkl_file)
# ru.print_column_shapes(dff)
print("dff.shape", dff.shape)
print("dff.columns", dff.columns)

print("has_nearby", dff["has_nearby"].sum(), dff["has_nearby"].describe())

# print dff data for one (env, episode)
env_id = dff["env_id"].unique()[0]
episode = dff["episode_index"].unique()[0]
print(f"env_id: {env_id}, episode: {episode}")
dff_env_episode = dff[(dff["env_id"] == env_id) & (dff["episode_index"] == episode)]
print("dff_env_episode.shape", dff_env_episode.shape)
dff_env_episode[
    [
        "time_step",
        "agent_id",
        "position",
        "orientation",
        #    'was_bitten_by_agent_id',
        #    'bite_other_fish',
        "eating_event",
        "displacement",
    ]
].to_csv("one_episode.csv", index=False)

# print(dff[["rnn_states", "observations"]].head())

# Behavior and observations

In [ ]:
try:
    m_idx, a_idx, k_idx = ru.get_sensor_indices(dff)
    print(m_idx, a_idx, k_idx)
except Exception as e:
    try:
        m_idx, a_idx, k_idx = get_sensor_indices_from_cfg()
        m_idx = slice(m_idx[0], a_idx[0])  # Mormyromast sensors
        a_idx = slice(a_idx[0], k_idx[0])  # Ampullary sensors
        k_idx = slice(k_idx[0], k_idx[-1] + 1)  # Knollen sensors
        print(m_idx, a_idx, k_idx)
    except Exception as e2:
        print(
            "Error getting sensor indices from data or config. Please check the data format."
        )
        print("Error message:", e)
        print("Error message from config:", e2)
        sys.exit(1)


observation_width = dff["observations"].values[0].shape[0]
sensor_types = [
    ("Mormyromast", m_idx),
    ("Ampullary", a_idx),
    ("Knollen", k_idx),
    ("Other", slice(k_idx.stop, observation_width)),
]

In [ ]:
dff = add_attention_entropy_columns(dff, sensor_types)

entropy_columns = [col for col in dff.columns if "entropy" in col]
for col in entropy_columns:
    dff[col].hist(bins=50, figsize=(7, 4), alpha=0.5, label=col)
    plt.title(f"Entropy Distribution for Attention (Sub)Masks")
    plt.xlabel("Entropy [bits]")
    plt.ylabel("Frequency")
plt.legend(entropy_columns)
plt.tight_layout()

In [ ]:
# Behavior Densities
if "behavior_densities_1d" in analyses_to_run:
    dff, bin_labels = bin_agent_size(dff, num_bins=3, bin_column="size_bin")
    plot_behavior_densities_1d(dff)

In [ ]:
if "plot_sensor_topK_pca_components" in analyses_to_run:
    dff, bin_labels = bin_agent_size(dff, num_bins=3, bin_column="size_bin")
    print(bin_labels)

    # Plot sensor PCA components
    K = 4  # Number of top components to plot
    plot_sensor_topK_pca_components(dff, sensor_types, K=K)
    
    # Plot sensor PCA components by size
    plot_sensor_topK_pca_components_by_size(dff, sensor_types, K=4, bin_column="size_bin")


## Observation vector PCAs -- All observations and per-sensor type

In [ ]:
if "pca_rnn_obs_mask" in analyses_to_run:
    # Define your input variants
    variants = {
        "Observations": np.vstack(dff["observations"].tolist()),
        "Attention Mask": np.vstack(dff["attn_mask"].tolist()),
        "Obs * Mask": np.vstack((dff["observations"] * dff["attn_mask"]).tolist()),
    }

    # Run PCA analysis for each variant
    for label, matrix in variants.items():
        print(f"Performing PCA analysis for: {label}")
        perform_pca_analysis_rnn_obs_mask(matrix, title_prefix=label, sensor_types=sensor_types)

# RNN activity 

## PCAs -- Agent-specific and Pooled

In [ ]:
if "pca_individual_agent" in analyses_to_run:
    dff = dff.sort_values(by=["env_id", "agent_id", "episode_index", "time_step"])

    # Individual PCA analysis
    agent_ids = dff["agent_id"].unique().tolist()
    for agent_id in agent_ids[:1]:  # Loop through some or all agents
        dff_agent = dff[dff["agent_id"] == agent_id]
        rnn_states = np.vstack(dff_agent["rnn_states"].tolist())

        pca_full = PCA()
        rnn_states_pca = pca_full.fit_transform(rnn_states)
        plot_cumulative_variance(
            pca_full,
            f"Cumulative Variance Explained for Agent: {agent_id}",
            # knee_curve="concave",
            # knee_direction="increasing",
        )

        pca_2d = PCA(n_components=2)
        plot_pca_2d(
            pca_2d.fit_transform(rnn_states),
            dff_agent["agent_id"],
            pca_2d,
            title_string=f"Agent {agent_id}",
        )

        # pca_3d = PCA(n_components=3)
        # plot_pca_3d(pca_3d.fit_transform(rnn_states), dff_agent["agent_id"], pca_3d, agent_id)

In [ ]:
if "pca_pooled_agent" in analyses_to_run:
    # All agents on common PCA basis
    all_rnn_states = np.vstack(dff["rnn_states"].tolist())
    pca_common = PCA()
    rnn_states_pca = pca_common.fit_transform(all_rnn_states)

    plot_cumulative_variance(pca_common, "Cumulative Variance Explained Across All Agents")
    plot_all_agents_projected_2d(dff, pca_common)
    # plot_all_agents_projected_3d(dff, pca_common)

## Are RNN states & dynamics clumpy?

### Punctuated-ness of RNN states

In [ ]:
if "rnn_clumpiness_knn" in analyses_to_run:
    print(rnn_states_pca.shape)
    # WARNING: Slow -- takes almost 1.5 min for some of our experiments
    k_distances = knn_distance_analysis(
        rnn_states_pca,
        k=5,
        plot=True,
        pca_dim=50,
        verbose=True,
    )
    print(k_distances)

if "rnn_clumpiness_knn" in analyses_to_run:
    clumpiness_df = compute_clumpiness_scores_df(dff, window_size=10, k=5)
    all_clumpiness_scores = np.concatenate(clumpiness_df["clumpiness_score"].values)
    print("All clumpiness scores shape:", all_clumpiness_scores.shape)

    plt.figure(figsize=(8, 4))
    plt.hist(
        all_clumpiness_scores,
        bins=50,
        edgecolor="k",
        alpha=0.7,
    )
    plt.axvline(
        np.mean(all_clumpiness_scores), color="r", linestyle="dashed", linewidth=2
    )
    plt.axvline(
        np.median(all_clumpiness_scores), color="g", linestyle="dashed", linewidth=2
    )
    plt.title("Distribution of Clumpiness Scores")
    plt.xlabel("Clumpiness Score")
    plt.ylabel("Frequency")
    plt.legend(["Mean", "Median"])
    plt.grid(True)
    plt.tight_layout()
    plt.show()

### RNN velocities (deltas)

In [ ]:
if "rnn_delta_distribution" in analyses_to_run:
    # Calculate RNN deltas
    dff["rnn_deltas"] = get_rnn_state_deltas(dff)

    # Plot histograms of RNN deltas across all agents
    dff["rnn_deltas"].hist(bins=50, figsize=(8, 4), alpha=0.7)
    plt.axvline(np.mean(dff["rnn_deltas"]), color="r", linestyle="dashed", linewidth=2)
    plt.axvline(
        np.median(dff["rnn_deltas"]), color="g", linestyle="dashed", linewidth=2
    )
    plt.title("Distribution of RNN State |Deltas| Across All Agents")
    plt.xlabel("RNN State |Delta|")
    plt.ylabel("Frequency")
    plt.legend(["Mean", "Median"])
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# RNN + Behavior -- simultaneous traces

In [ ]:
# Arena-space distances vs. RNN activity-space distances
# plot_rnn_vs_arena_distances(dff)

In [ ]:
if "rnn_behavior_viz" in analyses_to_run:
    # Sample some (env, episode) pairs externally
    NUM_SAMPLES = 1
    sampled_keys = (
        dff[["env_id", "episode_index"]]
        .drop_duplicates()
        .sample(NUM_SAMPLES)
        .itertuples(index=False, name=None)
    )
    print(sampled_keys)

    # Plot them
    if not task == "2f1p":
        plot_behavior_and_rnn_state_trajectories(dff, episode_keys=sampled_keys)
    else:
        # TODO: get arena_size_predefined from dff
        plot_behavior_and_rnn_state_trajectories(
            dff, episode_keys=sampled_keys, arena_size_predefined=(300, 300)
        )

# RNN Representations 

## Feature extraction/check

In [ ]:
# print(dff.columns)

In [ ]:
event_counter_colnames = [col for col in dff.columns if col.startswith("time_since_")]
print("event_counter_colnames:")
for col in event_counter_colnames:
    print(col)

In [ ]:
REP_CANDIDATES = [
    # Actions
    "move_forward",
    "turn_angle"
    # Movement/action related
    "eating_event",
    "displacement",
    # Observation/Environment related
    "position_x",
    "position_y",
    "orientation",
      # TODO: Wall-related
    # TODO: Cons-EOD related -- Knollen EOD counter, EWMAs, nearest EOD counter, etc
    # RNN
    # RNN-velocity
] + event_counter_colnames

In [ ]:
import utils_decoding as du

columns_before = dff.columns.tolist()
dff, decoding_feature_names = du.prepare_features_for_decoding(
    dff,
    normalize=False,
    behavior_mode="foraging",
)

print("New columns:", set(dff.columns) - set(columns_before))
print("Existing columns:", set(dff.columns) & set(columns_before))

In [ ]:
# print("Number of features for decoding:", len(decoding_feature_names))
# print("Decoding feature names:", decoding_feature_names)

# New features only
decoding_based_features = [
    "error_angle",
    "mormyromast_field_magnitude",
    "mormyromast_field_magnitude_log",
    "mormyromast_field_direction",
    "ampullary_field_magnitude",
    "ampullary_field_magnitude_log",
    "ampullary_field_direction",
    "knollen_field_magnitude",
    "knollen_field_magnitude_log",
    "knollen_field_direction",
    "distance_to_wall",
    "approach_nearest_velocity",
    "p_eod_past",
    "p_eod_centered",
    "p_eod_future",
]

overlapping_features = set(decoding_based_features) & set(dff.columns)

print("Number of decoding-based features:", len(decoding_based_features))
print("Overlapping features with existing data:", len(overlapping_features))


# TODO: What are morm/knollen/amp/error angle features referenced to?

## Visualize the distributions of the regression features


In [ ]:
regression_features = list(set(REP_CANDIDATES + decoding_based_features))
regression_features = sorted(regression_features)
regression_features = [
    f for f in regression_features if f in dff.columns
]  # Only keep features that are present in the DataFrame
# missing_features = set(regression_features) - set(dff.columns)
bool_features = [f for f in regression_features if dff[f].dtype == "bool"]
for feature in bool_features:
    print(f"{feature} True: {dff[feature].mean() * 100:.2f}%")

count_features = [
    f
    for f in regression_features
    if dff[f].dtype == "int64" or dff[f].dtype == "int32"
]
# print([(f, dff[f].dtype) for f in regression_features if dff[f].dtype != "float64"])

In [ ]:
if "features_viz" in analyses_to_run:
    for feature in regression_features:
        if feature not in dff.columns:
            print(f"WARNING: Feature {feature} not found in DataFrame columns.")
            continue
        if dff[feature].dtype == "bool":
            continue
        else:
            # print(feature)
            # plt.figure(figsize=(4, 1.5))
            dff[feature].hist(bins=100, figsize=(4, 1.5), log=True)
            plt.title(feature)
            plt.show()

## Hypothesis -- smaller agents tend to spend more time hugging the wall


In [ ]:

# print("Percent NAs in arena_size:", dff["arena_size"].isna().mean() * 100)
hug_threshold = 3  # cm
dff["hugging_wall"] = dff["distance_to_wall"] < hug_threshold

# print(dff["distance_to_wall"].value_counts(bins=10, sort=False))
# Percent NAs
# print("Number of NAs in distance_to_wall:", dff["distance_to_wall"].isna().sum())
# print("Percent NAs in distance_to_wall:", dff["distance_to_wall"].isna().mean() * 100)

# Percent hugging wall
print(dff[["env_id", "episode_index", "agent_id", "time_step", "agent_size", "arena_size", "distance_to_wall", "hugging_wall"]].head())

# Compute fraction of hugging_wall for each (env_id, episode_index, agent_id)
hugging_summary = (
    dff.groupby(["env_id", "episode_index", "agent_id"])
    .agg({
        "hugging_wall": "mean",  # Fraction of time hugging wall
        "agent_size": "first"    # Assume constant agent_size within a trial
    })
    .reset_index()
)

print(hugging_summary.head())

# import matplotlib.pyplot as plt

# plt.scatter(
#     hugging_summary["agent_size"],
#     hugging_summary["hugging_wall"]
# )
# plt.xlabel("Agent Size")
# plt.ylabel("Fraction of Time Hugging Wall")
# plt.title("Trial-level Relationship: Agent Size vs Wall-Hugging")
# plt.show()

import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))

# Scatterplot
sns.scatterplot(
    x="agent_size",
    y="hugging_wall",
    data=hugging_summary,
    alpha=0.5
)

sns.regplot(
    x="agent_size",
    y="hugging_wall",
    data=hugging_summary,
    scatter=False,
    order=1  # 1 for linear, 2 for quadratic regression
)

plt.xlabel("Agent Size")
plt.ylabel(f"Fraction of Time Hugging Wall (< {hug_threshold} cm)")
plt.title("Smaller Agents Tend to Hug Walls More?")
plt.tight_layout()
plt.show()


import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 🧹 Bin agent_size into e.g., quartiles (you can adjust the number of bins!)
num_bins = 4
dff["agent_size_bin"] = pd.qcut(dff["agent_size"], q=num_bins, labels=[f"Q{i+1}" for i in range(num_bins)])

plt.figure(figsize=(8, 5))
sns.violinplot(
    x="agent_size_bin",
    y="distance_to_wall",
    data=dff,
    inner="box",  # Show a mini boxplot inside each violin
    cut=0         # Do not extend violins beyond min/max of data
)
plt.xlabel("Agent Size Quartile")
plt.ylabel("Distance to Wall (cm)")
plt.title("Distribution of Distance to Wall by Agent Size Quartile")
plt.tight_layout()
plt.show()



## Feature decoding from RNN states

In [ ]:
# import importlib
# import utils_decoding

# importlib.reload(utils_decoding)  # force reload every time
# from utils_decoding import generate_linear_regression_report

# if "decoding_legacy" in analyses_to_run:
#     all_linear_reg_reports, all_models_and_data = generate_linear_regression_report(
#         dff,
#         regression_features,
#         morm_range=(0, 10),
#         knollen_range=(10, 100),
#         min_samples=2,
#     )

#     for x in [all_linear_reg_reports, all_models_and_data]:
#         print(type(x))
        
#     print("all_linear_reg_reports.keys():", all_linear_reg_reports.keys())
#     print("all_models_and_data.keys():", all_models_and_data.keys())

## [Refactored] Decoding 

In [ ]:
import importlib
from re import S
import cfg

importlib.reload(cfg)  # force reload every time
from cfg import FEATURE_METADATA, EXCLUDE_FROM_DECODING
import utils_decoding

importlib.reload(utils_decoding)  # force reload every time
from utils_decoding import (
    get_sensor_ranges,
    run_modular_regression_pipeline,
    plot_perf_test_normalized,
)

print("regression_features: ", regression_features)

regression_features = [
    f for f in regression_features if f not in EXCLUDE_FROM_DECODING
]  # Exclude features that are not used for decoding

# regression_features = ["position"] # our only "vector feature"

sensor_ranges = get_sensor_ranges(
    morm_range=(0, 10),
    knollen_range=(10, 100),
)

sensor_ranges = sensor_ranges[:1] # TESTING

print("sensor_ranges:", sensor_ranges)

if "decoding_modular" in analyses_to_run:
    # circular_features = [
    #     cf
    #     for cf in regression_features
    #     if cf in FEATURE_METADATA and FEATURE_METADATA[cf]["feature_type"] == "circular"
    # ][:2]
    # probability_features = [
    #     cf
    #     for cf in regression_features
    #     if cf in FEATURE_METADATA and FEATURE_METADATA[cf]["feature_type"] == "probability"
    # ][:]
    # boolean_features = [
    #     cf
    #     for cf in regression_features
    #     if cf in FEATURE_METADATA and FEATURE_METADATA[cf]["feature_type"] == "boolean"
    # ][:]
    # count_features = [
    #     cf
    #     for cf in regression_features
    #     if cf in FEATURE_METADATA and FEATURE_METADATA[cf]["feature_type"] == "count"
    # ][:]
    # print("Circular features:", circular_features)

    reports, all_models_and_data = run_modular_regression_pipeline(
        dff,
        regression_features=regression_features,
        # regression_features=circular_features,
        # regression_features=probability_features,
        # regression_features=boolean_features,
        # regression_features=count_features,
        sensor_ranges=sensor_ranges,
        feature_metadata=FEATURE_METADATA,
        # behavior_mode="foraging", # TODO
    )



In [ ]:
len(sensor_ranges[:1])


In [ ]:
import utils_decoding

importlib.reload(utils_decoding)  # force reload every time
from utils_decoding import (
    get_sensor_ranges,
    run_modular_regression_pipeline,
    plot_perf_test_normalized,
)

importlib.reload(cfg)  # force reload every time
from cfg import FEATURE_TYPE_COLORMAP, FEATURE_METADATA


if "decoding_modular" in analyses_to_run:
    plot_perf_test_normalized(reports, 
                            #   outfile_base=os.path.join(OUTPUTS_FOLDER, "decoding_modular"),
                            outfile_base='./',
                            feature_metadata=FEATURE_METADATA, 
                            exclusion_list=EXCLUDE_FROM_DECODING, 
                            name_key=["name", "short_name"][0],
                            color_map=FEATURE_TYPE_COLORMAP,
                            sort_by_ftype=True,
                            ftype_order=["scalar", "vector", "circular", "probability", "boolean", "count", "other"],
                            )


## RNN-delta to feature correlations

In [ ]:
if "rnn_delta_corr" in analyses_to_run:
    # Calculate correlations of RNN deltas with a list of features
    feature_list = REP_CANDIDATES + decoding_based_features
    for pad_before in [True]: # False not needed
        print(f"Calculating RNN deltas with pad_before={pad_before}")
        dff["rnn_deltas"] = get_rnn_state_deltas(dff, pad_before=pad_before)
        feature_correlations = calculate_column_feature_correlations(
            dff,
            feature_list,
            col="rnn_deltas",
        )
        feature_correlations["abs_corr"] = feature_correlations["corr"].abs()
        print("RNN deltas correlations with features:")
        # print(rnn_deltas_correlations)
        feature_correlations.sort_values(by="abs_corr", ascending=False, inplace=True)
        print(feature_correlations.head(20))

    if False:  # Skip
        for feature in feature_list:
            plt.figure(figsize=(4, 2.5))
            plt.scatter(
                dff["rnn_deltas"],
                dff[feature],
                alpha=0.5,
                s=1,
                label=feature,
            )
            plt.xlabel("RNN Deltas")
            plt.ylabel("Feature Values")
            plt.title(f"RNN Deltas vs {feature}")
            plt.legend()
            plt.show()

In [ ]:
if "attn_entropy_corr" in analyses_to_run:
    # Calculate correlations of attention mask entropy with a list of features
    print("Calculating correlations of attention mask entropy with features...")
    # Entropy of Attention Masks
    feature_list = REP_CANDIDATES + decoding_based_features
    entropy_columns = [col for col in dff.columns if "entropy" in col]
    for feature in entropy_columns:
        feature_correlations = calculate_column_feature_correlations(
            dff, feature_list, col=feature
        )
        feature_correlations["abs_corr"] = feature_correlations["corr"].abs()
        feature_correlations.sort_values(by="abs_corr", ascending=False, inplace=True)
        print(f"{feature} correlations with features:")
        print(feature_correlations.head(10))
        print()

## UMAP on RNN states

In [ ]:
from utils_rnn import (
    get_umap_embedding,
    plot_bool_features_umap,
    plot_feature_umap,
    )

if "umap_bool_features" in analyses_to_run or "umap_non_bool_features" in analyses_to_run:
    umap_embedding = get_umap_embedding(dff, force_rerun = True, umap_embedding_file = "./umap_embedding.npy", do_plot=True)

    OUT_DIR = "./umap_feature_plots"
    os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
import importlib
import cfg
importlib.reload(cfg)  # force reload every time
from cfg import FEATURE_METADATA

if "umap_bool_features" in analyses_to_run:
    # Plot boolean features
    bool_features = [
        f for f in FEATURE_METADATA if FEATURE_METADATA[f].get("feature_type") == "boolean" 
    ]
    bool_exclusion_list = [
        "bite_action",
        "emit_eod",
        "has_agent_in_knollen_range",
        "has_food_in_food_sensing_range",
    ]
    bool_features = [f for f in bool_features if f not in bool_exclusion_list ]
    print("bool_features:", bool_features)

    print("Number of boolean features:", len(bool_features))
    print("Boolean features:", bool_features)
    plot_bool_features_umap(dff, umap_embedding, OUT_DIR, bool_features)

In [ ]:
if "umap_non_bool_features" in analyses_to_run:
    # Plot non-bool features
    feature_plot_config = FEATURE_METADATA

    progress = tqdm.tqdm(feature_plot_config.items(), desc="Plotting features")
    for feature, config in progress:
        progress.set_postfix(feature=feature)
        if feature not in dff.columns:
            print(f"Skipping {feature}: not in DataFrame")
            continue
        plot_feature_umap(dff, umap_embedding, feature, OUT_DIR, config, show=True)

<!-- ## RNN-activity spectrogram analysis -->
